---
title : "Dam Live"
---

In [ ]:
# initialiseer de (toolbox continu inzicht) modules
from pathlib import Path

from toolbox_continu_inzicht.base.config import Config
from toolbox_continu_inzicht.base.data_adapter import DataAdapter

## Inladen van belastingen FEWS HHNK

<details>
<summary>Configuratie</summary>

```yaml
GlobalVariables:
    rootdir: "data_sets/run_damlive"
    moments: [ 0, 24 ]
    calc_time: "2024-11-06 08:00:00"
    aquo_alias:
        H.meting: "WATHTE"
        WATHTE [m][NAP][OW]: "WATHTE"

    LoadsFews:
        host: "https://fews.hhnk.nl"
        port: 443
        region: "fewspiservice"
        version: "1.25"
        parameters: [ "H.meting", "WATHTE [m][NAP][OW]" ]

DataAdapter:
    default_options:
        csv:
            sep: ","
    locaties:
        type: csv
        path: "locations_fews.csv"
    waterstanden:
        type: csv
        path: "hidden_waterstanden_fews.csv"
    waterstanden_xml:
        type: xml_timeseries
        path: "waterstanden_fews.xml"
        parameter_mapping:
            WATHTE: H.meting
        location_mapping:
            MPN-N-24: BP

```

</details>

In [ ]:
config = Config(
    config_path=Path().cwd()
    / "data_sets"
    / "run_damlive"
    / "test_loads_fews_grondwater.yaml"
)
config.lees_config()

data_adapter = DataAdapter(config)

In [ ]:
from toolbox_continu_inzicht.loads.loads_fews.loads_fews import LoadsFews


loads_fews = LoadsFews(data_adapter=data_adapter)

In [ ]:
df_out = loads_fews.run(input="locaties", output="waterstanden_xml")

In [ ]:
data_adapter.input("waterstanden_xml")
file = data_adapter.config.data_adapters["waterstanden_xml"]["abs_path"]
with open(file) as f:
    xml_content = f.read()
# laast bovenste stukje van de xml zien
print(xml_content[:1300])

Voor het geval dat FEWS HHNK niet beschikbaar is, kan met het volgende bestand wel gerekend worden.
In het voorbeeld van FEWS hierboven is het een xml bestand maar elke tabel met meetreeks data kan gebruikt worden.

Bijvoorbeeld de csv:

```yaml
...
waterstanden:
    type: csv
    path: "waterstanden_fews.csv"
...
```

In [ ]:
data_adapter.input("waterstanden")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
df_waterstanden = data_adapter.input("waterstanden_xml")
for station in df_waterstanden["measurement_location_code"].unique():
    df_station = df_waterstanden[
        df_waterstanden["measurement_location_code"] == station
    ].copy()
    df_station.rename(columns={"value": station}, inplace=True)
    df_station.plot(x="date_time", y=station, ax=ax, legend=station)

## Runnen van dam_live

Voor dam live is een licentie van Deltares nodig.

Dit voorbeeld is gemaakt met een versie van damlive versie 26.1.0.7015.


In [1]:
# initialiseer de (toolbox continu inzicht) modules
from pathlib import Path

from toolbox_continu_inzicht.base.config import Config
from toolbox_continu_inzicht.base.data_adapter import DataAdapter

In [2]:
config = Config(
    config_path=Path().cwd() / "data_sets" / "run_damlive" / "run_damlive.yaml"
)
config.lees_config()

data_adapter = DataAdapter(config)

De benoemde waterstand metingen tabel moet mee gegeven worden

In [3]:
data_adapter.input("waterstanden_xml")

,date_time,measurement_location_code,parameter_code,unit,value
0,2024-11-06 08:00:00,BP,H.meting,m,-0.484
1,2024-11-06 09:00:00,BP,H.meting,m,-0.488
2,2024-11-06 10:00:00,BP,H.meting,m,-0.486
3,2024-11-06 11:00:00,BP,H.meting,m,-0.480
4,2024-11-06 12:00:00,BP,H.meting,m,-0.474
...,...,...,...,...,...
120,2024-11-07 04:00:00,PB-287,H.meting,m,-1.389
121,2024-11-07 05:00:00,PB-287,H.meting,m,-1.386
122,2024-11-07 06:00:00,PB-287,H.meting,m,-1.388
123,2024-11-07 07:00:00,PB-287,H.meting,m,-1.394


En de parameters moeten ook nog worden mee geven, in dit geval rekenen we met bishop

In [4]:
data_adapter.input("parameters_csv")

,parameter_names,parameter_values
0,CalculationModules_StabilityInside,1
1,CalculationModules_StabilityOutside,0
2,CalculationModules_PipingBligh,0
3,CalculationModules_PipingWti,0
4,StabilityParameters_CalculationModel,Bishop
5,StabilityParameters_SearchMethod,Grid


Daarnaast moet in de `root_dir` een `{project_naam}.damx` bestand staan wat uitvoer is van een [`Dam stabiliteit`](https://publicwiki.deltares.nl/spaces/DAM/pages/166462239/Aanmaken+DAM+Live+project) berekening, en een map met de stix bestanden: `{project_naam}.geometries2D` waar in het `.damx` bestand naar wordt verwezen. 

In dit voorbeeld is de `project_naam`:  `WV2030_Purmer`. 

Het projectnaam moet meegeven worden in de opties, 

In [5]:
from toolbox_continu_inzicht.dam_live import UpdateDamLive

update_dam_live = UpdateDamLive(data_adapter=data_adapter)

Dit kan 10-20 minuten duren: 

In [6]:
update_dam_live.run(input=["waterstanden_xml", "parameters_csv"], output="output_file")

In [7]:
update_dam_live.data_adapter.input("live.OutputTimeSeries")

,date_time,measurement_location_code,parameter_code,unit,value
0,2024-11-06 08:00:00,None,StabilityInsideFactor,m,1.047177
1,2024-11-06 09:00:00,None,StabilityInsideFactor,m,1.047054
2,2024-11-06 10:00:00,None,StabilityInsideFactor,m,1.047253
3,2024-11-06 11:00:00,None,StabilityInsideFactor,m,1.047337
4,2024-11-06 12:00:00,None,StabilityInsideFactor,m,1.047520
...,...,...,...,...,...
1595,2024-11-07 04:00:00,None,BishopCircleRadius,m,18.692500
1596,2024-11-07 05:00:00,None,BishopCircleRadius,m,18.692500
1597,2024-11-07 06:00:00,None,BishopCircleRadius,m,18.692500
1598,2024-11-07 07:00:00,None,BishopCircleRadius,m,18.692500
